# Experiment 9 - Retrieval-Augmented Generation (RAG) Document QA


In [ ]:
!pip -q install sentence-transformers faiss-cpu transformers sentencepiece

In [ ]:
# Small educational corpus based on the public BattingCoach README/project description.
# It summarizes project claims; it is not a substitute for the repository source or a coaching manual.
documents = [
 {"doc_id":"BC01","title":"Project overview","source":"BattingCoach README",
  "text":"BatCoach AI Pro is described as a browser-based cricket batting laboratory and virtual coach. It analyzes webcam video and provides feedback about batting technique."},
 {"doc_id":"BC02","title":"Pose and biomechanics","source":"BattingCoach README",
  "text":"The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its biomechanics module evaluates cues such as lead elbow elevation, lead knee flexion, head-over-knee balance, and spine forward lean."},
 {"doc_id":"BC03","title":"Practice modes","source":"BattingCoach README",
  "text":"The README describes shadow practice without a bat and live willow practice with a bat. Shadow practice bypasses bat-specific checks. Live willow mode includes bat tracking."},
 {"doc_id":"BC04","title":"Computer vision and shot classification","source":"BattingCoach README",
  "text":"The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a VideoMAE transformer for classifying cricket shots across ten foundational shot types."},
 {"doc_id":"BC05","title":"Feedback and repeated errors","source":"BattingCoach README",
  "text":"The project describes on-screen coaching cues and voice feedback. It tracks total swings and clean repetitions, and repeated technique errors can trigger an intervention that freezes the rep counter."},
 {"doc_id":"BC06","title":"Limitations and responsible use","source":"Notebook-authored evaluation note",
  "text":"Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence and the quality of the input video. A single recording cannot guarantee future match performance or diagnose a medical condition. Treat automated feedback as an aid for practice, not a replacement for a qualified coach."},
]
chunks = []
for d in documents:
    # short chunks preserve the source title and unique chunk id
    chunks.append({"chunk_id":d["doc_id"]+"-C1","title":d["title"],"source":d["source"],"text":d["text"]})
print("Documents:",len(documents),"Chunks:",len(chunks))
for c in chunks: print(c["chunk_id"],"|",c["title"],"|",c["text"][:90]+"...")

Documents: 6 Chunks: 6
BC01-C1 | Project overview | BatCoach AI Pro is described as a browser-based cricket batting laboratory and virtual coa...
BC02-C1 | Pose and biomechanics | The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its ...
BC03-C1 | Practice modes | The README describes shadow practice without a bat and live willow practice with a bat. Sh...
BC04-C1 | Computer vision and shot classification | The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a Video...
BC05-C1 | Feedback and repeated errors | The project describes on-screen coaching cues and voice feedback. It tracks total swings a...
BC06-C1 | Limitations and responsible use | Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence an...


In [ ]:
import numpy as np, faiss, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

EMBED_MODEL="sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL="google/flan-t5-small"
embedder=SentenceTransformer(EMBED_MODEL)
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=AutoTokenizer.from_pretrained(GEN_MODEL)
llm=AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL).to(device).eval()

emb=embedder.encode([c["text"] for c in chunks],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
index=faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print("Indexed",index.ntotal,"chunks. Generator:",GEN_MODEL,"on",device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (15 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Indexed 6 chunks. Generator: google/flan-t5-small on cpu


In [ ]:
def retrieve(question, k=3):
    q=embedder.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores, ids=index.search(q,k)
    return [(chunks[int(i)],float(s)) for s,i in zip(scores[0],ids[0]) if i >= 0]

def answer_rag(question,k=3):
    retrieved=retrieve(question,k)
    context="\n".join([f"[{c['chunk_id']} | {c['title']}] {c['text']}" for c,score in retrieved])
    prompt=("Answer the question using only the context below. If the context does not contain enough information, "
            "say that the answer cannot be determined from the provided documents. Do not invent facts.\n\n"
            f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:")
    inputs=tokenizer(prompt,return_tensors="pt",truncation=True,max_length=512).to(device)
    with torch.no_grad():
        out=llm.generate(**inputs,max_new_tokens=120,num_beams=2)
    answer=tokenizer.decode(out[0],skip_special_tokens=True)
    return answer,retrieved

test_questions = [
 "Which technology is described for pose estimation and kinematic angles?",
 "What is used for oriented bat tracking in live willow mode?",
 "How does the project respond to repeated technique errors?",
 "Explain the difference between shadow practice and live willow practice.",
 "What are two limitations of webcam-based feedback?",
 "What is the player's current batting average in the latest tournament? This information may not be in the corpus."
]
rag_records=[]
for q in test_questions:
    ans, ret=answer_rag(q,k=3)
    print("\nQUESTION:",q,"\nANSWER:",ans,"\nSOURCES:")
    for c,score in ret:
        print(f"- {c['chunk_id']} | {c['title']} | similarity={score:.3f}: {c['text']}")
    rag_records.append({"question":q,"answer":ans,"source_chunk_ids":", ".join(c["chunk_id"] for c,s in ret),
                        "retrieved_context":"\n".join(c["text"] for c,s in ret)})


QUESTION: Which technology is described for pose estimation and kinematic angles? 
ANSWER: MediaPipe 
SOURCES:
- BC02-C1 | Pose and biomechanics | similarity=0.642: The project describes MediaPipe-based pose estimation and kinematic angle extraction. Its biomechanics module evaluates cues such as lead elbow elevation, lead knee flexion, head-over-knee balance, and spine forward lean.
- BC06-C1 | Limitations and responsible use | similarity=0.386: Computer-vision feedback depends on camera angle, lighting, occlusion, model confidence and the quality of the input video. A single recording cannot guarantee future match performance or diagnose a medical condition. Treat automated feedback as an aid for practice, not a replacement for a qualified coach.
- BC04-C1 | Computer vision and shot classification | similarity=0.317: The project describes YOLOv8-OBB for oriented bat tracking in live willow mode and a VideoMAE transformer for classifying cricket shots across ten foundational shot typ

In [ ]:
import pandas as pd
rag_df=pd.DataFrame(rag_records)
display(rag_df)
rag_df.to_csv("experiment9_rag_test_results.csv",index=False)

,question,answer,source_chunk_ids,retrieved_context
0,Which technology is described for pose estimat...,MediaPipe,"BC02-C1, BC06-C1, BC04-C1",The project describes MediaPipe-based pose est...
1,What is used for oriented bat tracking in live...,YOLOv8-OBB,"BC03-C1, BC04-C1, BC01-C1",The README describes shadow practice without a...
2,How does the project respond to repeated techn...,freezes the rep counter. [BC05-C1 | Feedback a...,"BC05-C1, BC06-C1, BC01-C1",The project describes on-screen coaching cues ...
3,Explain the difference between shadow practice...,[BC03-C1 | Practice modes],"BC03-C1, BC04-C1, BC05-C1",The README describes shadow practice without a...
4,What are two limitations of webcam-based feedb...,Computer-vision feedback depends on camera ang...,"BC06-C1, BC05-C1, BC01-C1",Computer-vision feedback depends on camera ang...
5,What is the player's current batting average i...,[BC03-C1],"BC01-C1, BC04-C1, BC03-C1",BatCoach AI Pro is described as a browser-base...


In [ ]:
# Optional baseline comparison: direct LLM answer without retrieved context.
def answer_direct(question):
    prompt="Answer briefly and state uncertainty if unsure. Question: "+question
    inputs=tokenizer(prompt,return_tensors="pt",truncation=True,max_length=256).to(device)
    with torch.no_grad(): out=llm.generate(**inputs,max_new_tokens=100,num_beams=2)
    return tokenizer.decode(out[0],skip_special_tokens=True)

baseline_q=test_questions[-1]
print("Question:",baseline_q)
print("Direct LLM:",answer_direct(baseline_q))
print("RAG:",answer_rag(baseline_q)[0])

Question: What is the player's current batting average in the latest tournament? This information may not be in the corpus.
Direct LLM: 69.0
RAG: [BC03-C1]
